# Benchmarking Previous CAFA SOTA Models

## Models to retrain on our CAFA Data:
- InterLabelGO+ [github](https://github.com/QuanEvans/InterLabelGO)
    - Retrain on `train` split in `experiment_data` config and eval on the `validation` split
- NetGO4: (current SOTA on Cafa5) [site](https://dmiip.sjtu.edu.cn/ng4.0/#)
- SUPERMAGOv2 (another model) [paper](https://ieeexplore.ieee.org/document/11119523)

## Retrain InterLabelGO+

### Processing BioReason Data

In [ ]:
import pandas as pd
from datasets import load_dataset

# Load CAFA5 dataset
dataset = load_dataset("wanglab/cafa5", "experiment_data")

train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()

train_df_new = train_df[['protein_id', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc']]
val_df_new = val_df[['protein_id', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc']]

# Map columns to GO aspect codes
aspect_map = {
    'go_bp': 'BPO',
    'go_mf': 'MFO',
    'go_cc': 'CCO'
}

def process_dataframe(df):
    records = []

    for _, row in df.iterrows():
        entry_id = row['protein_id']

        for col, aspect in aspect_map.items():
            go_terms = row[col]

            # Skip None or NaN
            if go_terms is None:
                # print(f"None for {entry_id} {aspect}")
                continue
            if isinstance(go_terms, float) and pd.isna(go_terms):
                # print(f"NaN for {entry_id} {aspect}")
                continue

            # Normalize to list
            terms = go_terms

            # Append valid GO terms
            for term in terms:
                if term.startswith('GO:'):
                    records.append((entry_id, aspect, term))
    return pd.DataFrame(records, columns=['EntryID', 'aspect', 'term'])


# ---------- TRAIN ----------
train_terms_new = process_dataframe(train_df_new)
train_terms_new.to_csv("Data/cafa5_raw_data/train_terms_new.tsv", sep="\t", index=False)

# Write FASTA
with open("Data/cafa5_raw_data/train_seq_new.fasta", "w") as f:
    for _, row in train_df_new.iterrows():
        f.write(f">{row['protein_id']}\n{row['sequence']}\n")

print(f"✅ Saved {len(train_terms_new)} train terms and sequences successfully.")


# ---------- VALIDATION ----------
val_terms_new = process_dataframe(val_df_new)
val_terms_new.to_csv("Data/cafa5_raw_data/val_terms_new.tsv", sep="\t", index=False)

# Write FASTA
with open("Data/cafa5_raw_data/val_seq_new.fasta", "w") as f:
    for _, row in val_df_new.iterrows():
        f.write(f">{row['protein_id']}\n{row['sequence']}\n")

print(f"✅ Saved {len(val_terms_new)} validation terms and sequences successfully.")

## Evaluating Retrained Interlabel+ on our Val Data

In [ ]:
#!/usr/bin/env python3
"""
Script to evaluate the trained InterLabelGO+ model on test set
Calculates Fmax, precision, recall, and other metrics similar to training validation
"""

import os
import argparse
import pandas as pd
import numpy as np
import torch
import pickle
from tqdm import tqdm
import scipy.sparse as ssp

from Network.model_utils import FmaxMetric
from utils import obo_tools
from settings import settings_dict as settings

# Initialize OBO tools
oboTools = obo_tools.ObOTools(
    go_obo=settings['obo_file'],
    obo_pkl=settings['obo_pkl_file']
)

def read_ia(filename):
    """Read information content file"""
    ia_dict = dict()
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip().split()
            if len(line) != 2:
                raise ValueError('IA file format error')
            ia_dict[line[0]] = line[1]
    return ia_dict

def read_term_list(file_name):
    """Read GO term list"""
    term_list = []
    with open(file_name) as f:
        for line in f:
            term_list.append(line.rstrip())
    return term_list

def get_vec2go(term_list):
    """Create mapping from vector index to GO term"""
    vec2go_dict = dict()
    for i, go_term in enumerate(term_list):
        vec2go_dict[i] = go_term
    return vec2go_dict

def calculate_weight_matrix(term_list, vec2go_dict, ia_dict):
    """Calculate weight matrix based on information content"""
    weight_array = np.zeros(len(term_list))
    for i in range(len(term_list)):
        go_term = vec2go_dict[i]
        ia_score = ia_dict.get(go_term, 0)
        weight_array[i] = float(ia_score) + 1e-16
    return weight_array

def load_predictions(prediction_file):
    """Load predictions from TSV file"""
    df = pd.read_csv(prediction_file, sep='\t')
    return df

def load_ground_truth(ground_truth_file):
    """Load ground truth from TSV file"""
    df = pd.read_csv(ground_truth_file, sep='\t')
    return df

def create_prediction_matrix(predictions_df, term_list, protein_ids):
    """Convert predictions to matrix format"""
    # Create mapping from GO term to index
    term_to_idx = {term: idx for idx, term in enumerate(term_list)}
    protein_to_idx = {protein: idx for idx, protein in enumerate(protein_ids)}
    
    # Initialize prediction matrix
    pred_matrix = np.zeros((len(protein_ids), len(term_list)))
    
    # Fill prediction matrix
    for _, row in predictions_df.iterrows():
        protein_id = row['EntryID']
        term = row['term']
        score = row['score']
        
        if protein_id in protein_to_idx and term in term_to_idx:
            pred_matrix[protein_to_idx[protein_id], term_to_idx[term]] = score
    
    return pred_matrix

def create_ground_truth_matrix(ground_truth_df, term_list, protein_ids):
    """Convert ground truth to matrix format"""
    # Create mapping from GO term to index
    term_to_idx = {term: idx for idx, term in enumerate(term_list)}
    protein_to_idx = {protein: idx for idx, protein in enumerate(protein_ids)}
    
    # Initialize ground truth matrix
    gt_matrix = np.zeros((len(protein_ids), len(term_list)))
    
    # Fill ground truth matrix
    for _, row in ground_truth_df.iterrows():
        protein_id = row['EntryID']
        term = row['term']
        
        if protein_id in protein_to_idx and term in term_to_idx:
            gt_matrix[protein_to_idx[protein_id], term_to_idx[term]] = 1
    
    return gt_matrix

def evaluate_aspect(predictions_df, ground_truth_df, aspect, term_list, ia_dict, device='cuda'):
    """Evaluate predictions for a specific aspect"""
    print(f"Evaluating aspect: {aspect}")
    
    # Filter data for current aspect
    aspect_pred = predictions_df[predictions_df['aspect'] == aspect].copy()
    aspect_gt = ground_truth_df[ground_truth_df['aspect'] == aspect].copy()
    
    if aspect_pred.empty or aspect_gt.empty:
        print(f"No data found for aspect {aspect}")
        return None
    
    # Get unique protein IDs
    pred_proteins = set(aspect_pred['EntryID'].unique())
    gt_proteins = set(aspect_gt['EntryID'].unique())
    common_proteins = list(pred_proteins.intersection(gt_proteins))
    
    print(f"Found {len(common_proteins)} common proteins for aspect {aspect}")
    
    if len(common_proteins) == 0:
        print(f"No common proteins found for aspect {aspect}")
        return None
    
    # Create matrices
    pred_matrix = create_prediction_matrix(aspect_pred, term_list, common_proteins)
    gt_matrix = create_ground_truth_matrix(aspect_gt, term_list, common_proteins)
    
    # Convert to tensors
    pred_tensor = torch.from_numpy(pred_matrix).float().to(device)
    gt_tensor = torch.from_numpy(gt_matrix).float().to(device)
    
    # Calculate weight matrix
    vec2go = get_vec2go(term_list)
    weight_array = calculate_weight_matrix(term_list, vec2go, ia_dict)
    weight_tensor = torch.from_numpy(weight_array).to(device)
    
    # Calculate child matrix
    child_array = oboTools.generate_child_matrix(term_list)
    child_tensor = torch.from_numpy(child_array).to(device)
    
    # Initialize metric calculator
    metric = FmaxMetric(weight_matrix=weight_tensor, child_matrix=child_tensor, device=device)
    
    # Calculate metrics
    fmax, precision, recall, threshold = metric.compute_protein_centric_fm(
        pred_tensor, gt_tensor, margin=0.01, weight_tensor=weight_tensor, parent_prop=False
    )
    
    results = {
        'aspect': aspect,
        'fmax': fmax.item(),
        'precision': precision.item(),
        'recall': recall.item(),
        'threshold': threshold.item(),
        'num_proteins': len(common_proteins),
        'num_terms': len(term_list)
    }
    
    return results

def main():
    parser = argparse.ArgumentParser(description='Evaluate InterLabelGO+ predictions on test set')
    parser.add_argument('--predictions', type=str, required=True, 
                       help='Path to prediction TSV file (e.g., InterLabelGO+.tsv)')
    parser.add_argument('--ground_truth', type=str, 
                       default=settings['train_terms_tsv'].replace('train_terms.tsv', 'val_terms_new.tsv'),
                       help='Path to ground truth TSV file')
    parser.add_argument('--aspects', type=str, nargs='+', default=['BPO', 'CCO', 'MFO'],
                       choices=['BPO', 'CCO', 'MFO'], help='Aspects to evaluate')
    parser.add_argument('--device', type=str, default='cuda', help='Device to use (cuda/cpu)')
    parser.add_argument('--output', type=str, default='evaluation_results.txt', 
                       help='Output file for results')
    
    args = parser.parse_args()
    
    # Check if CUDA is available
    if args.device == 'cuda' and not torch.cuda.is_available():
        print("CUDA not available, using CPU")
        args.device = 'cpu'
    
    # Load data
    print("Loading predictions...")
    predictions_df = load_predictions(args.predictions)
    print(f"Loaded {len(predictions_df)} predictions")
    
    print("Loading ground truth...")
    ground_truth_df = load_ground_truth(args.ground_truth)
    print(f"Loaded {len(ground_truth_df)} ground truth annotations")
    
    # Load IA dictionary
    print("Loading information content...")
    ia_dict = read_ia(settings['ia_file'])
    
    # Load term lists for each aspect
    term_lists = {}
    for aspect in args.aspects:
        term_file = os.path.join(settings['TRAIN_DATA_CLEAN_DIR'], aspect, 'term_list.txt')
        if os.path.exists(term_file):
            term_lists[aspect] = read_term_list(term_file)
            print(f"Loaded {len(term_lists[aspect])} terms for {aspect}")
        else:
            print(f"Warning: Term list not found for {aspect} at {term_file}")
    
    # Evaluate each aspect
    all_results = []
    for aspect in args.aspects:
        if aspect not in term_lists:
            print(f"Skipping {aspect} - no term list available")
            continue
            
        results = evaluate_aspect(
            predictions_df, ground_truth_df, aspect, 
            term_lists[aspect], ia_dict, args.device
        )
        
        if results is not None:
            all_results.append(results)
            print(f"{aspect}: Fmax={results['fmax']:.4f}, "
                  f"Precision={results['precision']:.4f}, "
                  f"Recall={results['recall']:.4f}, "
                  f"Threshold={results['threshold']:.3f}")
    
    # Calculate overall metrics
    if all_results:
        overall_fmax = np.mean([r['fmax'] for r in all_results])
        overall_precision = np.mean([r['precision'] for r in all_results])
        overall_recall = np.mean([r['recall'] for r in all_results])
        
        print(f"\nOverall Results:")
        print(f"Fmax: {overall_fmax:.4f}")
        print(f"Precision: {overall_precision:.4f}")
        print(f"Recall: {overall_recall:.4f}")
        
        # Save results
        with open(args.output, 'w') as f:
            f.write("InterLabelGO+ Test Set Evaluation Results\n")
            f.write("=" * 50 + "\n\n")
            
            for result in all_results:
                f.write(f"Aspect: {result['aspect']}\n")
                f.write(f"  Fmax: {result['fmax']:.4f}\n")
                f.write(f"  Precision: {result['precision']:.4f}\n")
                f.write(f"  Recall: {result['recall']:.4f}\n")
                f.write(f"  Threshold: {result['threshold']:.3f}\n")
                f.write(f"  Proteins: {result['num_proteins']}\n")
                f.write(f"  Terms: {result['num_terms']}\n\n")
            
            f.write(f"Overall Results:\n")
            f.write(f"  Fmax: {overall_fmax:.4f}\n")
            f.write(f"  Precision: {overall_precision:.4f}\n")
            f.write(f"  Recall: {overall_recall:.4f}\n")
        
        print(f"Results saved to {args.output}")
    else:
        print("No valid results to save")

if __name__ == '__main__':
    main()


In [ ]:
python predict.py -w our_val_data_results -f Data/cafa5_raw_data/val_seq_new.fasta --use_gpu

python evaluate_test_set.py \
    --predictions our_val_data_results/InterLabelGO+.tsv \
    --ground_truth Data/cafa5_raw_data/val_terms_new.tsv \
    --output evaluation_results.txt 

```
InterLabelGO+ Test Set Evaluation Results
==================================================

Aspect: BPO
  Fmax: 0.5518
  Precision: 0.5388
  Recall: 0.5655
  Threshold: 0.360
  Proteins: 8506
  Terms: 4407

Aspect: CCO
  Fmax: 0.6812
  Precision: 0.6826
  Recall: 0.6798
  Threshold: 0.350
  Proteins: 9045
  Terms: 1014

Aspect: MFO
  Fmax: 0.7354
  Precision: 0.7526
  Recall: 0.7189
  Threshold: 0.380
  Proteins: 7673
  Terms: 1435

Overall Results:
  Fmax: 0.6561
  Precision: 0.6580
  Recall: 0.6547
```